In [154]:
import pandas as pd
import numpy as np
import os, sys
from tqdm.notebook import tqdm
from pathlib import Path
from plotly import express as px
import plotly.graph_objects as go
from sklearn.linear_model import  LinearRegression,Ridge
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from copy import copy

In [25]:
class Estimator:
    def __init__(self):
        pass

    def fit(self):
        pass

    def __call__(self, well_data:pd.DataFrame, typewell_data:pd.DataFrame)->pd.Series:
        """Returns a TVT pd.Series"""
        pass

class Evaluator:
    def __init__(self):
        pass

    def evaluate_estimator(self, estimator:Estimator, file_path:Path=Path("rogii-wellbore-geology-prediction/train",), p=0.8):
        well_files={
            f.name.replace("__horizontal_well.csv",""):{"Well":f,"TypeWell":f.parent/f.name.replace("__horizontal_well.csv","__typewell.csv")}
            for f in file_path.glob("*__horizontal_well.csv")
            }
        filenames = np.random.permutation(list(well_files.keys()))
        train_files = {well_files[f] for f in filenames[:round(len(well_files)*p)]}
        test_files = {well_files[f] for f in filenames[round(len(well_files)*p):]}
        tvt_trues, tvt_preds= {}, {}
        estimator.fit(train_files)
        for fname, paths in tqdm(test_files.items()):
            well_data, typewell_data = pd.read_csv(paths["Well"]), pd.read_csv(paths["TypeWell"])
            first_pred_idx = well_data.TVT_input.shift(1).dropna().index[-1]
            tvt_trues[fname] = well_data.loc[first_pred_idx:,"TVT"]
            tvt_preds[fname] = estimator(well_data)
        tvt_trues, tvt_preds = pd.DataFrame(tvt_trues), pd.DataFrame(tvt_preds)
        tvt_errors = tvt_preds - tvt_trues
        rmse = np.nanmean(tvt_errors.values.flatten()**2)**0.5
        return tvt_trues, tvt_preds, tvt_errors, rmse
    
    def make_submission(self, estimator:Estimator, file_path:Path=Path("rogii-wellbore-geology-prediction/test"),submission_file="submission.csv"):
        well_files={
            f.name.replace("__horizontal_well.csv",""):{"Well":f,"TypeWell":f.parent/f.name.replace("__horizontal_well.csv","__typewell.csv")}
            for f in file_path.glob("*__horizontal_well.csv")
            }
        estimator.fit(well_files)
        tvt_preds = []
        for fname, paths in tqdm(well_files.items()):
            well_data, typewell_data = pd.read_csv(paths["Well"]), pd.read_csv(paths["TypeWell"])
            tvt_pred = estimator(well_data).rename("tvt")
            tvt_pred.index = pd.Series(fname + "_"+tvt_pred.index.values.astype(str),name="id")
            tvt_preds.append(tvt_pred)
        tvt_preds = pd.concat(tvt_preds,axis=0)
        tvt_preds.to_csv(submission_file)
        

In [331]:
class LinearMappingEstimator(Estimator):
    def __init__(self):
        self.mapping_data = None
        self.positional_cols = ["X","Y"]
        self.formation_cols = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']

    def fit(self, well_files):
        self.mapping_data = pd.concat([
            pd.read_csv(well_files[f]["Well"]).loc[:,
                self.positional_cols + self.formation_cols                                                  
                ] for f in tqdm(well_files.keys())
            ],axis=0).dropna()
        self.formation_model = KNeighborsRegressor(n_neighbors=15, weights="uniform").fit(X=self.mapping_data[self.positional_cols], y=self.mapping_data[self.formation_cols])

    def predict_formations(self, well_data):
        return pd.DataFrame(
            self.formation_model.predict(X=well_data.loc[:,self.positional_cols]), columns=self.formation_cols, index=well_data.index
            )



In [332]:
file_path = Path("rogii-wellbore-geology-prediction/train",)
well_files={
            f.name.replace("__horizontal_well.csv",""):{"Well":f,"TypeWell":f.parent/f.name.replace("__horizontal_well.csv","__typewell.csv")}
            for f in file_path.glob("*__horizontal_well.csv")
            }

In [333]:
file = list(well_files.keys())[105]
well_data = pd.read_csv(well_files[file]["Well"])
first_pred_idx = well_data.TVT_input.shift(1).dropna().index[-1]
last_known_idx = well_data.TVT_input.dropna().index[-1]

In [334]:
tfiles = copy(well_files)
del tfiles[file]

In [335]:
self = LinearMappingEstimator()
self.fit(tfiles)

  0%|          | 0/772 [00:00<?, ?it/s]

In [336]:
predicted_formation_height = self.predict_formations(well_data)

In [337]:
z_change = well_data.loc[first_pred_idx:,"Z"]-well_data.loc[last_known_idx,"Z"]
pred_formation_change = predicted_formation_height.loc[first_pred_idx:] - predicted_formation_height.loc[last_known_idx]

In [338]:
px.scatter(predicted_formation_height)

In [339]:
px.scatter(pred_formation_change.subtract(z_change,axis=0).median(axis=1))

In [340]:
px.scatter(well_data.loc[first_pred_idx:,"TVT"] - well_data.loc[last_known_idx,"TVT"])

In [341]:
px.scatter(well_data.loc[first_pred_idx:,"BUDA"] -well_data.loc[last_known_idx,"BUDA"])

In [342]:
px.scatter(well_data.loc[first_pred_idx:,"Z"] -well_data.loc[last_known_idx,"Z"])

In [343]:
px.scatter((well_data.loc[first_pred_idx:,"BUDA"] -well_data.loc[last_known_idx,"BUDA"])-(well_data.loc[first_pred_idx:,"Z"] -well_data.loc[last_known_idx,"Z"]))